# Stage 6b — Prior Admission ICD Carry-Forward

**Input** : Previous admissions' `ground_truth.txt` files  
**Output**: `stage_06b_history_context/history_codes.json` per admission

## What this stage does
For each admission, collect all ICD-10 codes from the **same patient's prior admissions**
and carry them forward as candidate codes for the current admission.

These codes are already verified — no UMLS lookup, no grounding, no scoring needed.
Chronic conditions (diabetes, cirrhosis, CKD, etc.) tend to persist across admissions,
so prior codes are a strong signal for what will appear on the current discharge summary.

If a patient has only one admission, no history codes are available and this stage
contributes nothing.

## Pipeline position
```
Stage 5: Evidence Scoring + Pruning        ✅
Stage 6: SNOMED Grounding (active)         ✅
Stage 6b: Prior Admission Carry-Forward    ← THIS NOTEBOOK
Stage 7:  Final ICD Decision               (next)
```

## 1. Setup

In [1]:
import json
from pathlib import Path
import pandas as pd

RECORDS_DIR    = Path(r'C:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records')
STAGE6_OUTPUT  = 'stage_06_snomed_grounding'
STAGE6B_OUTPUT = 'stage_06b_history_context'

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith('patient_')])
print(f'Patients found: {len(patients)}')

Patients found: 15


## 2. Ground Truth Parser

In [2]:
def parse_ground_truth(gt_path):
    """Parse ground_truth.txt → list of {icdCode, title} dicts."""
    if not gt_path.exists():
        return []
    codes = []
    with open(gt_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and line[0].isdigit() and '\u2014' in line:
                parts    = line.split('\u2014', 1)
                raw_code = parts[0].strip()
                title    = parts[1].strip() if len(parts) > 1 else ''
                code = raw_code.split('.')[-1].strip().split()[0].upper()
                if code:
                    codes.append({'icdCode': code, 'title': title})
    return codes


print('Ground truth parser defined.')

Ground truth parser defined.


## 3. Collect Prior Admission Codes for All Patients

In [3]:
all_history = []

for patient_dir in patients:
    patient_id = patient_dir.name.replace('patient_', '')
    adm_root   = patient_dir / 'admissions'
    adm_dirs   = sorted(adm_root.iterdir()) if adm_root.exists() else []

    for i, adm_dir in enumerate(adm_dirs):
        # Collect ICD codes from all admissions that came before this one
        prior_codes = {}  # icdCode -> {title, source_admissions}
        for prior_dir in adm_dirs[:i]:
            gt_codes = parse_ground_truth(prior_dir / 'ground_truth.txt')
            for entry in gt_codes:
                code = entry['icdCode']
                if code not in prior_codes:
                    prior_codes[code] = {
                        'icdCode'          : code,
                        'title'            : entry['title'],
                        'source_admissions': [],
                    }
                prior_codes[code]['source_admissions'].append(prior_dir.name.replace('hadm_', ''))

        # Sort by frequency — codes seen in more prior admissions are more likely chronic
        prior_list = sorted(prior_codes.values(), key=lambda x: len(x['source_admissions']), reverse=True)

        output = {
            'patient_id'        : patient_id,
            'admission_id'      : adm_dir.name.replace('hadm_', ''),
            'n_prior_admissions': i,
            'n_prior_codes'     : len(prior_list),
            'prior_icd_codes'   : prior_list,
        }

        out_dir = adm_dir / STAGE6B_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / 'history_codes.json', 'w') as f:
            json.dump(output, f, indent=2)

        all_history.append(output)
        print(f'Patient {patient_id} | {adm_dir.name} | '
              f'prior_admissions={i} prior_codes={len(prior_list)}')

print(f'\nDone. {len(all_history)} admissions processed.')

Patient 10361982 | hadm_24286431 | prior_admissions=0 prior_codes=0
Patient 10426859 | hadm_29908281 | prior_admissions=0 prior_codes=0
Patient 10458324 | hadm_21744342 | prior_admissions=0 prior_codes=0
Patient 11251337 | hadm_29568708 | prior_admissions=0 prior_codes=0
Patient 11474876 | hadm_29672491 | prior_admissions=0 prior_codes=0
Patient 11607177 | hadm_23293838 | prior_admissions=0 prior_codes=0
Patient 12007928 | hadm_23749816 | prior_admissions=0 prior_codes=0
Patient 13196707 | hadm_21475988 | prior_admissions=0 prior_codes=0
Patient 13508515 | hadm_21834271 | prior_admissions=0 prior_codes=0
Patient 13952483 | hadm_23852410 | prior_admissions=0 prior_codes=0
Patient 16014068 | hadm_29042843 | prior_admissions=0 prior_codes=0
Patient 17774110 | hadm_27339772 | prior_admissions=0 prior_codes=0
Patient 18412100 | hadm_26093939 | prior_admissions=0 prior_codes=0
Patient 19104262 | hadm_24271247 | prior_admissions=0 prior_codes=0
Patient 19632936 | hadm_26696232 | prior_admissi

## 4. Inspect One Admission

In [10]:
EXAMPLE_IDX = 5
ex = all_history[EXAMPLE_IDX]

print(f'Patient          : {ex["patient_id"]}')
print(f'Admission        : {ex["admission_id"]}')
print(f'Prior admissions : {ex["n_prior_admissions"]}')
print(f'Prior ICD codes  : {ex["n_prior_codes"]}')
print()

if ex['prior_icd_codes']:
    for c in ex['prior_icd_codes'][:20]:
        freq = len(c['source_admissions'])
        print(f'  {c["icdCode"]:10s}  x{freq}  {c["title"][:60]}')
else:
    print('  No prior admissions — first admission for this patient.')

Patient          : 11607177
Admission        : 23293838
Prior admissions : 0
Prior ICD codes  : 0

  No prior admissions — first admission for this patient.


## 5. Evaluate — Stage 6 + Prior Codes Combined

In [6]:
def load_ground_truth(patient_dir, hadm_id):
    gt_path = patient_dir / 'admissions' / f'hadm_{hadm_id}' / 'ground_truth.txt'
    return [e['icdCode'] for e in parse_ground_truth(gt_path)]


def _prefix_match(pred, gt_set):
    if pred in gt_set:
        return True
    for gt in gt_set:
        if gt.startswith(pred) or pred.startswith(gt):
            return True
    return False


def _f1(tp, pred, gt):
    p = len(tp) / len(pred) if pred else 0.0
    r = len(tp) / len(gt)   if gt   else 0.0
    return round(2*p*r/(p+r) if (p+r) > 0 else 0.0, 3)


all_metrics = []

for hist in all_history:
    pid  = hist['patient_id']
    hadm = hist['admission_id']
    pdir = RECORDS_DIR / f'patient_{pid}'
    gt   = load_ground_truth(pdir, hadm)
    if not gt:
        continue
    gt_set = set(gt)

    # Stage 6 codes
    s6_path  = pdir / 'admissions' / f'hadm_{hadm}' / STAGE6_OUTPUT / 'grounded_symptoms.json'
    s6_codes = set()
    if s6_path.exists():
        with open(s6_path) as f:
            s6 = json.load(f)
        for b in s6.get('grounded_branches', []):
            for s in b.get('grounded_symptoms', []):
                for c in s.get('icd_candidates', []):
                    s6_codes.add(c['icdCode'].replace('.', '').upper())

    # Stage 6b — prior admission codes (already verified, no transformation needed)
    s6b_codes = {e['icdCode'] for e in hist['prior_icd_codes']}

    combined = s6_codes | s6b_codes

    tp_s6   = {p for p in s6_codes  if _prefix_match(p, gt_set)}
    tp_s6b  = {p for p in s6b_codes if _prefix_match(p, gt_set)}
    tp_comb = {p for p in combined  if _prefix_match(p, gt_set)}

    all_metrics.append({
        'patient_id'        : pid,
        'n_prior_admissions': hist['n_prior_admissions'],
        'n_gt'              : len(gt_set),
        'n_s6'              : len(s6_codes),
        'f1_s6'             : _f1(tp_s6,   s6_codes,  gt_set),
        'n_s6b'             : len(s6b_codes),
        'f1_s6b'            : _f1(tp_s6b,  s6b_codes, gt_set),
        'n_combined'        : len(combined),
        'f1_combined'       : _f1(tp_comb, combined,   gt_set),
    })

df   = pd.DataFrame(all_metrics)
cols = ['patient_id', 'n_prior_admissions', 'n_gt', 'n_s6', 'f1_s6', 'n_s6b', 'f1_s6b', 'n_combined', 'f1_combined']
print(df[cols].to_string(index=False))
print(f'\nMean F1 — Stage 6 only       : {df["f1_s6"].mean():.3f}')
print(f'Mean F1 — Prior codes only   : {df["f1_s6b"].mean():.3f}')
print(f'Mean F1 — Combined 6 + prior : {df["f1_combined"].mean():.3f}')

patient_id  n_prior_admissions  n_gt  n_s6  f1_s6  n_s6b  f1_s6b  n_combined  f1_combined
  10361982                   0     5     0  0.000      0     0.0           0        0.000
  10426859                   0    22    10  0.000      0     0.0          10        0.000
  10458324                   0    12    30  0.238      0     0.0          30        0.238
  11251337                   0     7    12  0.000      0     0.0          12        0.000
  11474876                   0    17     0  0.000      0     0.0           0        0.000
  11607177                   0    13     0  0.000      0     0.0           0        0.000
  12007928                   0    19     5  0.000      0     0.0           5        0.000
  13196707                   0    32    11  0.047      0     0.0          11        0.047
  13508515                   0    14     5  0.000      0     0.0           5        0.000
  13952483                   0    25    20  0.178      0     0.0          20        0.178
  16014068

## 6. Save Summary

In [7]:
summary = {
    'stage'             : 'stage_06b_prior_admission_carry_forward',
    'n_admissions'      : len(all_history),
    'mean_f1_s6'        : round(df['f1_s6'].mean(), 3),
    'mean_f1_s6b'       : round(df['f1_s6b'].mean(), 3),
    'mean_f1_combined'  : round(df['f1_combined'].mean(), 3),
    'stage5_baseline_f1': 0.028,
    'per_patient'       : all_metrics,
}

out_path = RECORDS_DIR / 'stage_06b_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'Saved: {out_path}')
print(f'\nStage 5 baseline F1       : {summary["stage5_baseline_f1"]}')
print(f'Stage 6 mean F1           : {summary["mean_f1_s6"]}')
print(f'Prior codes mean F1       : {summary["mean_f1_s6b"]}')
print(f'Combined 6 + prior F1     : {summary["mean_f1_combined"]}')

Saved: C:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records\stage_06b_summary.json

Stage 5 baseline F1       : 0.028
Stage 6 mean F1           : 0.106
Prior codes mean F1       : 0.0
Combined 6 + prior F1     : 0.106
